# Introduction train model

# Add root to sys.path

In [ ]:
import os
import sys

# Add the root directory to the Python path
module_dir = os.path.abspath('../..')
if module_dir not in sys.path:
    sys.path.append(module_dir)
for x in sys.path:
    print(x)

# Imports

In [ ]:
from pathlib import Path

import torch

from notebooks.create_settings.Settings import Settings

from source.shared import Encoder
from source.create_model import create_model
from source.shared import load_model
from source.train_model import train_model
from source.evaluate_model import ModelEvaluator
from source.shared import SyntaxDeriverValidationReporter
from source.train_model import plot_correct
from source.shared import generate_predicted_dictum

# Settings

In [ ]:
import panel as pn
pn.extension()
pn.config.sizing_mode="stretch_width"

settings = Settings()
settings.view()

# Load model

In [ ]:
# load model
def set_up_model() -> Path:
    model_folder_path = Path(settings.model_folder_path)
    model_name = 'model.pt'
    model_file_path = model_folder_path.joinpath(model_name).resolve()
    if model_file_path.exists():
        print(f"model_file already exists: {model_file_path}")
    else:
        create_model(settings=settings)
    return model_file_path

model_file_path = set_up_model()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.backends.mps.is_available():
    device = "mps"
print(f'device={device}')
encoder = Encoder.load_from_json(corpus_folder_path=settings.corpus_folder_path)
print(f'loading model and optimizer from checkpoint={model_file_path}')
model, optimizer = load_model(model_checkpoint_path=model_file_path, device=device, encoder=encoder)

# Train model

In [ ]:
# train model
max_train_epochs = 10 * 10 * 10 * 10 * 1
train_model(model, optimizer, max_train_epochs=max_train_epochs, corpus_folder_path=settings.corpus_folder_path, model_folder_path=settings.model_folder_path)

# Evaluate model

In [ ]:
%%time
def print_context(context):
    parts = context.split('\n')
    if len(parts) >= 2:
        print(f'prompt: {parts[0]}')
        print(f'predicted_statement: {parts[1]}')

def evaluate_model(model, max_examples: int, max_print_error: int, max_print_ok: int):
    model_evaluator = ModelEvaluator(corpus_folder_path=settings.corpus_folder_path, model=model)
    model_evaluator.evaluate_model(max_examples=max_examples)
    syntax_deriver_db= model_evaluator.syntax_deriver.syntax_deriver_db
    validation_reporter = SyntaxDeriverValidationReporter(syntax_deriver_db=syntax_deriver_db, block_size=settings.block_size)
    validation_reporter.print_validation_report(max_print_error=max_print_error, max_print_ok=max_print_ok, print_context=print_context)

    # print(f'\n=== Show all tables ===')
    # syntax_deriver = model_evaluator.syntax_deriver
    # syntax_deriver.syntax_deriver_db.show_math_statements_table()
    # syntax_deriver.syntax_deriver_db.show_rule_errors_table()
    # print(f'\n=== Show main_view ===')
    # syntax_deriver.syntax_deriver_db.create_main_view()
    # syntax_deriver.syntax_deriver_db.print_main_view()

max_examples = 10 * 10 * 1 * 1
max_print_error = 3
max_print_ok = 0
if max_examples > 0:
    evaluate_model(model=model, max_examples=max_examples, max_print_error=max_print_error, max_print_ok=max_print_ok)

# Plot

In [ ]:
plot_correct(model_folder_path=settings.model_folder_path, bucket_count=50, xlabel='train step', ylabel='percent correct', title='assert logic')

# New statement percentage

In [ ]:
%%time
def get_set_of_train_examples():
    corpus_file_path = Path(settings.corpus_folder_path).joinpath('corpus.txt').resolve()
    with open(corpus_file_path, 'r') as file:
        train_examples = file.read().split('\n')
    set_of_train_examples = set(train_examples) # used to see if the inference is in the train examples
    return set_of_train_examples

def print_new_statement_percentage(max_statement_count: int):
    _ = model.eval() # put model in inference mode (not training mode)
    new_count = 0
    prompt = '|- '
    terminal_token = '<|over|>'
    set_of_train_examples = get_set_of_train_examples() # used to see if the inference is in the train examples
    for _ in range(max_statement_count):
        predicted_dictum = generate_predicted_dictum(prompt=prompt, terminal_token=terminal_token, model=model)
        if predicted_dictum not in set_of_train_examples:
            new_count += 1
    print(f'new_statement_percentage={100 * new_count / max_statement_count: .0f}%')

print_new_statement_percentage(max_statement_count=10 * 10)